# QHBERT Full Comparison — combines qiskit-practice.ipynb + qhbert_end_to_end.ipynb

One notebook, one dataset (ISOT), everything reported side by side:

1. **Tier 1/2 recap** — the tutorial-style Qiskit `VQC` (`zz_feature_map`/`pauli_feature_map` +
   `real_amplitudes`/`efficient_su2`) on TF-IDF+SVD features, from `qiskit-practice.ipynb`.
   Loaded here from its saved results file, not re-run.
2. **QHBERT** — frozen DistilBERT CLS embedding -> classical bridge -> quantum layer -> head,
   from `qhbert_end_to_end.ipynb`'s architecture, generalized to a wider trainable-parameter
   budget and run with **both** quantum backends (PennyLane `TorchLayer` and Qiskit
   `EstimatorQNN`+`TorchConnector`), across a small qubit/layer ablation grid, with an
   optional **Zero-Noise Extrapolation (Mitiq)** pass.
3. **Classical baselines** — TF-IDF+SVM, frozen-DistilBERT+bridge+head with the quantum layer
   *removed* (isolates what quantum adds), and the PyTorch `BiLSTM`/`CNN`/`Transformer`
   baselines from `src/models/baselines.py`.

All on the same ISOT split (`clmentbisaillon/fake-and-real-news-dataset`,
80/20, label convention `0=fake, 1=real` — matching `qiskit-practice.ipynb`,
**not** `qhbert_end_to_end.ipynb`'s `0=real, 1=fake`; watch this if comparing
raw numbers against that notebook directly).

**Running locally (this version), not on Kaggle:**
- Needs `dataset/Fake.csv` and `dataset/True.csv` at the repo root (same files
  `qiskit-practice.ipynb` uses) — `find_local`/`require_local` in Cells 2/5
  check both the repo root and one level up, so it works whether this kernel's
  working directory is the repo root or `kaggle/`.
- No CUDA GPU was detected on this machine — `DEVICE` resolves to `"cpu"`
  automatically (Cell 4), so DistilBERT embedding extraction (Cell 6) and the
  classical baselines (Cell 16) will run on CPU and be slower than on a Kaggle
  GPU session; the quantum training itself (Cells 7-11) is CPU-bound either
  way, unaffected.
- If you ran `qiskit-practice.ipynb` in this same working directory and it
  wrote `isot_comparison_results.json`, Cell 2 picks it up automatically;
  otherwise that section is skipped with a printed note (not a silent gap).

In [1]:
# Cell 1 — installs + library check (fail loudly here, not deep into a training loop)
# Running locally (not Kaggle): installs into whichever Python this kernel uses; harmless/idempotent
# if already present.
# !pip install -q pennylane pennylane-qiskit qiskit qiskit-machine-learning qiskit-aer mitiq

import importlib


def check_lib(name):
    try:
        mod = importlib.import_module(name)
        print(f"  {name}: OK ({getattr(mod, '__version__', 'unknown')})")
        return True
    except ImportError as e:
        print(f"  {name}: MISSING ({e})")
        return False


print("Library check:")
_all_ok = True
for lib in ["torch", "transformers", "pennylane", "qiskit", "qiskit_machine_learning",
            "qiskit_aer", "mitiq", "sklearn"]:
    _all_ok &= check_lib(lib)

if not _all_ok:
    print("\nWARNING: one or more libraries failed to import — fix before running the training cells below.")
else:
    print("\nAll libraries OK.")

Library check:
  torch: OK (2.10.0+cu128)
  transformers: OK (5.0.0)
  pennylane: MISSING (No module named 'pennylane')
  qiskit: MISSING (No module named 'qiskit')
  qiskit_machine_learning: MISSING (No module named 'qiskit_machine_learning')
  qiskit_aer: MISSING (No module named 'qiskit_aer')
  mitiq: MISSING (No module named 'mitiq')
  sklearn: OK (1.6.1)



In [2]:
# Cell 2 — path helper (this notebook lives in kaggle/, but the repo root — where
# dataset/ and isot_comparison_results.json live — may or may not be the working
# directory depending on how the kernel was launched) + Tier 1/2 recap
import json
import os


def find_local(relative_to_repo_root):
    """Try relative_to_repo_root as-is, then one level up (covers both
    'launched from repo root' and 'launched from kaggle/' cases)."""
    for candidate in (relative_to_repo_root, os.path.join("..", relative_to_repo_root)):
        if os.path.exists(candidate):
            return candidate
    return None  # caller decides whether missing is fatal or just "skip this part"


TIER12_RESULTS_PATH = find_local("isot_comparison_results.json")

tier12_results = None
if TIER12_RESULTS_PATH:
    with open(TIER12_RESULTS_PATH) as f:
        tier12_results = json.load(f)
    print(f"Loaded Tier 1/2 results from {TIER12_RESULTS_PATH}")
    print(f"  VQC test_acc: {tier12_results['vqc']['test_acc']:.3f}")
    print(f"  SVC (same features) test_acc: {tier12_results['svc_reduced']['test_acc']:.3f}")
    print(f"  SVC (full TF-IDF) test_acc: {tier12_results['svc_full_tfidf']['test_acc']:.3f}")
else:
    print("NOTE: isot_comparison_results.json not found (checked cwd and one level up) — run "
          "qiskit-practice.ipynb's Tier 1/2 cells first to include those results in the final "
          "combined table. Continuing without them (not a silent gap: this print is the record of it).")

NOTE: isot_comparison_results.json not found (checked cwd and one level up) — run qiskit-practice.ipynb's Tier 1/2 cells first to include those results in the final combined table. Continuing without them (not a silent gap: this print is the record of it).


In [ ]:
# Cell 3 — config
MAX_LENGTH = 512
BATCH_SIZE = 128  # larger than the original 32: PennyLane's backprop path amortizes per-call
                  # overhead better at larger batch (measured ~35% faster per-sample at 128 vs 32)
EPOCHS = 8        # down from a naive 15 — see the perf note in Cell 7. Even after the backprop
                  # fix, one PennyLane config over the full ~32K-row training set is
                  # ~15-20 min/epoch; 8 epochs x 2 configs is already ~4-5h, a large chunk of a
                  # 12h Kaggle session once DistilBERT extraction + baselines + ZNE are added.
LR_CLASSICAL = 1e-3
LR_QUANTUM = 5e-3  # was 1e-2 — SPSA-estimated gradients run larger-magnitude than backprop's;
                   # halved to reduce overshoot/oscillation on the Qiskit path

# Bridge/head widths — wider than QHBERTCore's fixed 768->64->8 (~50K params total) to use the
# Kaggle GPU quota for something; params live in bridge/head width, not qubit count (simulation
# cost there is exponential). ~150K-300K params keeps QHBERT ~350-700x smaller than BERT's 110M,
# which is the efficiency story the project's own reference doc leans on.
BRIDGE_DIMS = (256, 64)
HEAD_DIM = 64

# Ablation grid: backend x qubits x layers. Both backends now train on the FULL training set
# (PennyLane via backprop makes this cheap; Qiskit via SPSA makes it expensive — see
# QISKIT_TRAIN_SUBSAMPLE below). Qiskit configs are still capped at fewer qubits/layers than
# PennyLane's, since simulation cost is exponential in qubit count regardless of backend.
ABLATION_GRID = [
    {"backend": "pennylane", "n_qubits": 8, "n_layers": 3},
    {"backend": "pennylane", "n_qubits": 12, "n_layers": 3},
    {"backend": "qiskit", "n_qubits": 4, "n_layers": 2},
    {"backend": "qiskit", "n_qubits": 8, "n_layers": 2},
]
QISKIT_TRAIN_SUBSAMPLE = None  # None = train on the FULL set (len(train_ds), same as PennyLane), no
                                # subsampling. Previously 8000 (before that, 3000) purely to keep SPSA's
                                # per-batch cost tractable — at full scale this is a real, large time
                                # cost (SPSA measured ~20-30x slower per row than PennyLane's backprop
                                # path), so Cell 10's "time 1 epoch before committing" check is not
                                # optional here — always run it first and look at the projected total
                                # before launching the full ABLATION_GRID in Cell 11.
QISKIT_EPOCHS = 25             # was 15 — too few steps for SPSA's noisy gradient to converge

RUN_ZNE_ABLATION = True  # operates on a 200-sample test subset only (Cell 12/13) — cheap regardless
ZNE_NOISE_SCALE_FACTORS = [1.0, 2.0, 3.0]  # Richardson extrapolation, per the reference doc

# --- Full BERT fine-tune config (new pipeline, added after the frozen-embedding grid below) ---
# Unlike the frozen-embedding path above, DistilBERT is trainable here and runs live inside the
# training loop every step (no pre-cached embeddings — gradients must flow back into BERT), so
# these constants are deliberately smaller/cheaper than the frozen-embedding ones.
FT_MAX_LENGTH = 256  # halved from MAX_LENGTH=512: keeps full-BERT-fine-tune batches GPU-memory-safe
                      # and ~2x faster per step; title + lede (first ~256 tokens) carries most of the
                      # fake/real signal on ISOT anyway
FT_BATCH_SIZE = 16   # much smaller than BATCH_SIZE=128 — every step now runs DistilBERT forward+
                      # backward (66M trainable params), not just the bridge/quantum/head (~215K)
FT_EPOCHS = 3         # full fine-tuning converges faster and overfits an easy dataset like ISOT
                      # quickly; EPOCHS=8 (tuned for the frozen path) would waste session time here
LR_BERT = 2e-5        # standard BERT fine-tuning LR (Devlin et al.) — far below LR_CLASSICAL/
                      # LR_QUANTUM since BERT is pretrained and needs small updates, not
                      # trained-from-scratch-sized ones
FT_BACKEND = "pennylane"  # Qiskit's SPSA path combined with a live, trainable BERT is impractically
                          # slow for a full run (SPSA is already ~20-30x slower than PennyLane's
                          # backprop even with frozen/cached embeddings) — PennyLane is the practical
                          # default; Qiskit remains available via QHBERTFineTuneModel(backend="qiskit")
                          # if you want to try it on a small epoch count
FT_N_QUBITS, FT_N_LAYERS = 12, 3  # the ablation grid's best-performing PennyLane config (Cell 10's
                                   # sanity check on frozen embeddings: val_f1=0.9823 in 1 epoch)

In [4]:
!pip install -q pennylane

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 72.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 75.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 101.3 MB/s eta 0:00:0000:0100:01


In [5]:
# Cell 4 — imports
import glob
import re
import time

import numpy as np
import pandas as pd
import pennylane as qml
import torch
import torch.nn as nn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from torch.utils.data import DataLoader, TensorDataset
from transformers import DistilBertModel, DistilBertTokenizerFast

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cuda


In [ ]:
# Cell 5 — load ISOT locally (same source + label convention as qiskit-practice.ipynb: 0=fake,
# 1=real), from dataset/ (see find_local in Cell 2 for the path resolution). Carve a validation
# split out of the 80% train portion so best-checkpoint selection has a val set.


def clean_text(text: str) -> str:
    text = re.sub(r"<.*?>", " ", str(text))
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def require_local(relative_to_repo_root):
    path = find_local(relative_to_repo_root)
    if path is None:
        raise FileNotFoundError(
            f"Could not find '{relative_to_repo_root}' in the current directory or one level up "
            f"— make sure dataset/Fake.csv and dataset/True.csv exist at the repo root."
        )
    return path


df_fake = pd.read_csv(require_local("dataset/Fake.csv"))
df_real = pd.read_csv(require_local("dataset/True.csv"))
df_fake["label"] = 0
df_real["label"] = 1

df = pd.concat([df_real, df_fake], axis=0)
df["content"] = (df["title"].astype(str) + " " + df["text"].astype(str)).apply(clean_text)
df = df[["content", "label"]].sample(frac=1, random_state=42).reset_index(drop=True)

X_train_text, X_test_text, y_train_full, y_test = train_test_split(
    df["content"], df["label"], test_size=0.2, random_state=42, stratify=df["label"]
)
# carve validation out of the 80% train portion (90/10) -> ~32,326 train / ~3,592 val, test untouched
X_train_text, X_valid_text, y_train, y_valid = train_test_split(
    X_train_text, y_train_full, test_size=0.1, random_state=42, stratify=y_train_full
)

print(f"Train: {len(X_train_text)}, Valid: {len(X_valid_text)}, Test: {len(X_test_text)}")
print(f"Label balance (train): {y_train.value_counts().to_dict()}")

In [7]:
# Cell 6 — frozen DistilBERT embedding extraction (~44,898 rows total). No CUDA GPU detected
# on this machine (DEVICE resolved to "cpu" in Cell 4) -> this will be noticeably slower than
# on a Kaggle GPU session; expect this cell alone to take a while (rough ballpark: tens of
# minutes, depends on your CPU). MAX_LENGTH=512 is the biggest lever if you want it faster —
# drop it in Cell 3 (e.g. to 256) at the cost of truncating longer articles more aggressively.
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")
bert = DistilBertModel.from_pretrained("distilbert-base-uncased").to(DEVICE)
bert.eval()


@torch.no_grad()
def extract_embeddings(texts):
    all_embeddings = []
    texts = list(texts)
    for i in range(0, len(texts), BATCH_SIZE):
        batch = texts[i:i + BATCH_SIZE]
        encoded = tokenizer(batch, return_tensors="pt", padding=True,
                             truncation=True, max_length=MAX_LENGTH).to(DEVICE)
        out = bert(**encoded)
        all_embeddings.append(out.last_hidden_state[:, 0, :].cpu())
        if (i // BATCH_SIZE) % 10 == 0:
            print(f"  {i}/{len(texts)} embedded")
    return torch.cat(all_embeddings, dim=0)


splits_text = {"train": (X_train_text, y_train), "valid": (X_valid_text, y_valid), "test": (X_test_text, y_test)}
cached = {}
for split_name, (texts, labels) in splits_text.items():
    print(f"Extracting embeddings for {split_name} ({len(texts)} rows)...")
    embeddings = extract_embeddings(texts)
    cached[split_name] = {"embeddings": embeddings, "labels": torch.tensor(labels.values, dtype=torch.long)}

torch.save(cached, "isot_full_embeddings.pt")
print("Saved isot_full_embeddings.pt")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Extracting embeddings for train (32326 rows)...
  0/32326 embedded
  1280/32326 embedded
  2560/32326 embedded
  3840/32326 embedded
  5120/32326 embedded
  6400/32326 embedded
  7680/32326 embedded
  8960/32326 embedded
  10240/32326 embedded
  11520/32326 embedded
  12800/32326 embedded
  14080/32326 embedded
  15360/32326 embedded
  16640/32326 embedded
  17920/32326 embedded
  19200/32326 embedded
  20480/32326 embedded
  21760/32326 embedded
  23040/32326 embedded
  24320/32326 embedded
  25600/32326 embedded
  26880/32326 embedded
  28160/32326 embedded
  29440/32326 embedded
  30720/32326 embedded
  32000/32326 embedded
Extracting embeddings for valid (3592 rows)...
  0/3592 embedded
  1280/3592 embedded
  2560/3592 embedded
Extracting embeddings for test (8980 rows)...
  0/8980 embedded
  1280/8980 embedded
  2560/8980 embedded
  3840/8980 embedded
  5120/8980 embedded
  6400/8980 embedded
  7680/8980 embedded
  8960/8980 embedded
Saved isot_full_embeddings.pt


## QHBERT model — both quantum backends

Same architecture as `src/models/quantum_layers.py` + `src/models/qhbert.py`
(`QHBERTModel`), reproduced inline so this notebook stays self-contained on
Kaggle (matching `qhbert_end_to_end.ipynb`'s own convention) rather than
depending on the repo being present as a Kaggle dataset/utility script.

In [ ]:
# Cell 7 — quantum backend factories (mirrors src/models/quantum_layers.py)
#
# IMPORTANT perf note (measured locally, 12 qubits/3 layers/batch 32):
#   - PennyLane, diff_method="parameter-shift" + per-sample Python loop: ~95s/batch
#     (~26 HOURS/epoch on the full training set) — unusable, and parameter-shift
#     actually raises NotImplementedError on a batched call at all (broadcasted
#     tapes + trainable weights isn't supported, PennyLane issue #4462).
#   - PennyLane, diff_method="backprop" + a single batched call: ~1.3s/batch (~22 min/epoch).
#     Backprop is also the numerically right choice here — this is a noiseless
#     simulator, so there's nothing parameter-shift buys you over exact backprop.
#   - Qiskit's EstimatorQNN has no backprop-equivalent. Its default gradient
#     (ParamShiftEstimatorGradient) didn't finish a single batch of 32 in several
#     minutes at 12 qubits. Switching to SPSAEstimatorGradient (O(1) evaluations
#     per step instead of O(2 x num_params)) brought it to ~36s/batch at 12q/3L,
#     ~7s/batch at 8q/3L, ~1.4s/batch at 4q/2L — still far slower than PennyLane,
#     which is why the ablation grid in Cell 3 caps Qiskit at fewer qubits.


def build_pennylane_layer(n_qubits, n_layers):
    dev = qml.device("default.qubit", wires=n_qubits)

    @qml.qnode(dev, interface="torch", diff_method="backprop")
    def circuit(inputs, weights):
        qml.AngleEmbedding(inputs, wires=range(n_qubits), rotation="Y")
        for layer in range(n_layers):
            for i in range(n_qubits):
                qml.CNOT(wires=[i, (i + 1) % n_qubits])
            for qubit in range(n_qubits):
                qml.RY(weights[layer, qubit, 0], wires=qubit)
                qml.RZ(weights[layer, qubit, 1], wires=qubit)
        return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

    weight_shapes = {"weights": (n_layers, n_qubits, 2)}
    return qml.qnn.TorchLayer(circuit, weight_shapes)


def build_qiskit_layer(n_qubits, n_layers, feature_map="zz", ansatz="real_amplitudes"):
    from qiskit.circuit.library import efficient_su2, pauli_feature_map, real_amplitudes, zz_feature_map
    from qiskit.primitives import StatevectorEstimator
    from qiskit.quantum_info import SparsePauliOp
    from qiskit_machine_learning.connectors import TorchConnector
    from qiskit_machine_learning.gradients import SPSAEstimatorGradient
    from qiskit_machine_learning.neural_networks import EstimatorQNN

    feature_maps = {"zz": zz_feature_map, "pauli": pauli_feature_map}
    ansatze = {"real_amplitudes": real_amplitudes, "efficient_su2": efficient_su2}

    fmap = feature_maps[feature_map](feature_dimension=n_qubits, reps=1)
    ansatz_circuit = ansatze[ansatz](num_qubits=n_qubits, reps=n_layers)
    circuit = fmap.compose(ansatz_circuit)

    observables = [SparsePauliOp("I" * i + "Z" + "I" * (n_qubits - i - 1)) for i in range(n_qubits)]
    estimator = StatevectorEstimator()
    gradient = SPSAEstimatorGradient(estimator=estimator, epsilon=0.01, batch_size=10)
    # BUG FIX: batch_size was implicitly 1 -> a single random-perturbation gradient
    # estimate per call. Because input_gradients=True, that same noisy estimate is
    # ALSO used as the input Jacobian backpropagated into the classical bridge, so the
    # noise didn't stay contained to the quantum weights -- it corrupted the bridge's
    # gradient too, via the chain rule. This was the root cause of the collapsed
    # (val_f1=0.0000 every epoch) Qiskit runs. Averaging 10 perturbations per call
    # cuts gradient variance by ~sqrt(10) at a proportional cost in estimator calls.
    qnn = EstimatorQNN(
        circuit=circuit,
        observables=observables,
        input_params=list(fmap.parameters),
        weight_params=list(ansatz_circuit.parameters),
        input_gradients=True,  # required: gradients must flow back into the classical bridge
        estimator=estimator,
        gradient=gradient,
    )
    return TorchConnector(qnn)


def build_quantum_layer(backend, n_qubits, n_layers, **kwargs):
    if backend == "pennylane":
        return build_pennylane_layer(n_qubits, n_layers)
    if backend == "qiskit":
        return build_qiskit_layer(n_qubits, n_layers, **kwargs)
    raise ValueError(f"Unknown backend: {backend!r}")

In [ ]:
# Cell 8 — QHBERTModel (mirrors src/models/qhbert.py::QHBERTModel)


class QHBERTModel(nn.Module):
    def __init__(self, bert_dim=768, bridge_dims=BRIDGE_DIMS, n_qubits=8, n_layers=3,
                 backend="pennylane", head_dim=HEAD_DIM, num_labels=2, dropout=0.3,
                 quantum_kwargs=None, use_quantum=True):
        super().__init__()
        bridge_layers = []
        in_dim = bert_dim
        for dim in bridge_dims:
            bridge_layers += [nn.Linear(in_dim, dim), nn.LayerNorm(dim), nn.ReLU(), nn.Dropout(dropout)]
            in_dim = dim
        bridge_layers.append(nn.Linear(in_dim, n_qubits))
        self.bridge = nn.Sequential(*bridge_layers)

        # use_quantum=False -> classical-only ablation (isolates the quantum layer's contribution)
        self.use_quantum = use_quantum
        if use_quantum:
            self.quantum = build_quantum_layer(backend, n_qubits, n_layers, **(quantum_kwargs or {}))

        self.classifier = nn.Sequential(
            nn.Linear(n_qubits, head_dim), nn.ReLU(), nn.Dropout(dropout), nn.Linear(head_dim, num_labels),
        )

    def forward(self, cls_embedding):
        scaled = torch.tanh(self.bridge(cls_embedding)) * torch.pi
        if self.use_quantum:
            # batched call, NOT a per-sample Python loop: both TorchLayer and TorchConnector
            # accept a full (batch, n_qubits) tensor directly. Looping multiplies every
            # backward pass's cost by batch_size for nothing — measured ~74x slower on
            # PennyLane (12 qubits/3 layers/batch 32: ~95s looped vs ~1.3s batched).
            # The quantum simulator (default.qubit / StatevectorEstimator) is CPU-only, so on a
            # GPU run this must round-trip scaled -> cpu -> quantum -> back to scaled's original
            # device; .cpu()/.to() are autograd-differentiable, so gradients still flow correctly.
            features = self.quantum(scaled.cpu()).to(scaled.device)
        else:
            features = scaled
        return self.classifier(features)


def param_count(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def move_to_device_except_quantum(model, device):
    """model.to(device) would also move the quantum layer's weight parameter onto the GPU, but
    default.qubit/StatevectorEstimator expect CPU tensors internally — so move everything, then
    move just the quantum sub-module back to CPU. Used by both the frozen-embedding QHBERTModel
    (Cell 9) and the fine-tuned QHBERTFineTuneModel added later."""
    model.to(device)
    if hasattr(model, "quantum"):
        model.quantum.to("cpu")
    return model


print(f"Example param count (pennylane, 8 qubits): "
      f"{param_count(QHBERTModel(n_qubits=8, n_layers=3, backend='pennylane'))}")
print(f"Example param count (classical-only, no quantum layer): "
      f"{param_count(QHBERTModel(n_qubits=8, use_quantum=False))}")

In [ ]:
# Cell 9 — shared train/eval loop, used by every QHBERTModel config below. Bridge/classifier now
# actually run on DEVICE (cuda if available) — previously this loop never touched DEVICE at all,
# so training ran on CPU regardless of GPU availability. The quantum layer stays pinned to CPU
# inside forward() (see move_to_device_except_quantum / Cell 8) since default.qubit and
# StatevectorEstimator are CPU-only simulators.
train_ds = TensorDataset(cached["train"]["embeddings"], cached["train"]["labels"])
valid_ds = TensorDataset(cached["valid"]["embeddings"], cached["valid"]["labels"])
test_ds = TensorDataset(cached["test"]["embeddings"], cached["test"]["labels"])
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)


def evaluate(model, loader):
    model.eval()
    all_preds, all_probs, all_labels = [], [], []
    with torch.no_grad():
        for embeddings, labels in loader:
            logits = model(embeddings.to(DEVICE))
            probs = torch.softmax(logits, dim=1)[:, 1]
            all_preds.extend(logits.argmax(dim=1).cpu().tolist())
            all_probs.extend(probs.cpu().tolist())
            all_labels.extend(labels.tolist())
    return all_preds, all_probs, all_labels


def train_and_evaluate(model, run_name, epochs=EPOCHS, custom_train_loader=None):
    move_to_device_except_quantum(model, DEVICE)
    active_train_loader = custom_train_loader if custom_train_loader is not None else train_loader
    quantum_params = list(model.quantum.parameters()) if model.use_quantum else []
    classical_params = list(model.bridge.parameters()) + list(model.classifier.parameters())
    param_groups = [{"params": classical_params, "lr": LR_CLASSICAL}]
    if quantum_params:
        param_groups.append({"params": quantum_params, "lr": LR_QUANTUM})
    optimizer = torch.optim.Adam(param_groups)
    criterion = nn.CrossEntropyLoss()

    best_val_f1, best_state = 0.0, None
    start = time.time()
    for epoch in range(epochs):
        model.train()
        total_loss, n_seen = 0.0, 0
        for embeddings, labels in active_train_loader:
            embeddings, labels = embeddings.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(embeddings), labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * embeddings.size(0)
            n_seen += embeddings.size(0)

        val_preds, _, val_labels = evaluate(model, valid_loader)
        val_f1 = f1_score(val_labels, val_preds)
        print(f"[{run_name}] epoch {epoch + 1}/{epochs} — loss {total_loss / n_seen:.4f}, val_f1 {val_f1:.4f}")
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
    train_time = time.time() - start

    if best_state is not None:
        model.load_state_dict(best_state)
    test_preds, test_probs, test_labels = evaluate(model, test_loader)
    return {
        "run_name": run_name,
        "n_params": param_count(model),
        "train_time_sec": train_time,
        "accuracy": accuracy_score(test_labels, test_preds),
        "precision": precision_score(test_labels, test_preds),
        "recall": recall_score(test_labels, test_preds),
        "f1": f1_score(test_labels, test_preds),
        "auc_roc": roc_auc_score(test_labels, test_probs),
    }

In [ ]:
!pip install -q qiskit qiskit-machine-learning qiskit-aer

In [ ]:
# Cell 10 — runtime sanity check: time ONE epoch for the most expensive PennyLane config
# (full training set) and ONE epoch for the most expensive Qiskit config (now also the full
# training set, per QISKIT_TRAIN_SUBSAMPLE=None in Cell 3) before committing to the full grid —
# confirms the Cell 3 fixes actually worked, and gives a real projected total time, before you
# wait through the whole grid. Also builds qiskit_subsample_loader, reused by Cell 11.
_pl_largest = max([c for c in ABLATION_GRID if c["backend"] == "pennylane"], key=lambda c: c["n_qubits"])
print(f"Timing 1 epoch, full training set, largest PennyLane config: {_pl_largest}")
_m = QHBERTModel(n_qubits=_pl_largest["n_qubits"], n_layers=_pl_largest["n_layers"], backend="pennylane")
_t0 = time.time()
_ = train_and_evaluate(_m, run_name="sanity_check_pennylane", epochs=1)
_pl_epoch_time = time.time() - _t0
print(f"-> {_pl_epoch_time:.1f}s/epoch, projects to ~{_pl_epoch_time * EPOCHS / 60:.1f} min for "
      f"{EPOCHS} epochs on this config.")
del _m

if QISKIT_TRAIN_SUBSAMPLE is None:
    qiskit_subsample_loader = train_loader
    qiskit_train_rows = len(train_ds)
else:
    _qiskit_subset_idx = np.random.RandomState(0).choice(len(train_ds), size=QISKIT_TRAIN_SUBSAMPLE, replace=False)
    qiskit_subsample_loader = DataLoader(
        torch.utils.data.Subset(train_ds, _qiskit_subset_idx), batch_size=BATCH_SIZE, shuffle=True
    )
    qiskit_train_rows = QISKIT_TRAIN_SUBSAMPLE
_qk_largest = max([c for c in ABLATION_GRID if c["backend"] == "qiskit"], key=lambda c: c["n_qubits"])
print(f"\nTiming 1 epoch, {qiskit_train_rows}-row set, largest Qiskit config: {_qk_largest}")
_m = QHBERTModel(n_qubits=_qk_largest["n_qubits"], n_layers=_qk_largest["n_layers"], backend="qiskit")
_t0 = time.time()
_ = train_and_evaluate(_m, run_name="sanity_check_qiskit", epochs=1, custom_train_loader=qiskit_subsample_loader)
_qk_epoch_time = time.time() - _t0
print(f"-> {_qk_epoch_time:.1f}s/epoch, projects to ~{_qk_epoch_time * QISKIT_EPOCHS / 60:.1f} min for "
      f"{QISKIT_EPOCHS} epochs on this config.")
del _m

print("\nIf either projection is too slow for your remaining session time, shrink ABLATION_GRID, "
      "EPOCHS/QISKIT_EPOCHS, or QISKIT_TRAIN_SUBSAMPLE in Cell 3 before running Cell 11.")

In [ ]:
# Cell 11 — run the ablation grid (backend x qubits x layers), plus the classical-only
# (quantum-removed) ablation. PennyLane configs train on the full train_loader; Qiskit configs
# train on qiskit_subsample_loader (built in Cell 10) — which is the FULL train_loader too when
# QISKIT_TRAIN_SUBSAMPLE=None (see Cell 3), or an actual subsample otherwise.
qhbert_results = []
trained_models = {}  # keeps trained models around for the ZNE cell below

# quantum-removed ablation — isolates what the quantum layer actually contributes
classical_only_model = QHBERTModel(n_qubits=8, use_quantum=False)
qhbert_results.append(train_and_evaluate(classical_only_model, run_name="bridge_only_no_quantum"))
trained_models["bridge_only_no_quantum"] = classical_only_model

for config in ABLATION_GRID:
    run_name = f"{config['backend']}_{config['n_qubits']}q_{config['n_layers']}L"
    model = QHBERTModel(n_qubits=config["n_qubits"], n_layers=config["n_layers"], backend=config["backend"])
    if config["backend"] == "qiskit":
        result = train_and_evaluate(
            model, run_name=run_name, epochs=QISKIT_EPOCHS, custom_train_loader=qiskit_subsample_loader
        )
        result["train_rows"] = qiskit_train_rows
    else:
        result = train_and_evaluate(model, run_name=run_name, epochs=EPOCHS)
        result["train_rows"] = len(train_ds)
    result.update(config)
    qhbert_results.append(result)
    trained_models[run_name] = model

qhbert_results_df = pd.DataFrame(qhbert_results)
print("\nQHBERT ablation grid results (train_rows column says how much data each row used):")
print(qhbert_results_df.to_string(index=False))

## ±ZNE ablation

ZNE only means something on a **noisy** simulator — `default.qubit`/the
noiseless `StatevectorEstimator` above have nothing to mitigate. Scope, kept
deliberately narrow: take the best-performing **PennyLane** config's already-
trained weights from the grid above, and at **inference time only** (not
retrained with noise) compare three settings on a test subset:

1. noiseless (the original grid result, for reference)
2. noisy (`default.mixed` + depolarizing noise after every CNOT), unmitigated
3. the same noisy circuit, with Richardson-extrapolation ZNE applied

Training a fully noise-aware model end-to-end is real future work (per the
project's own reference doc) — this answers "does ZNE recover the accuracy
noise costs you," which is the ablation the doc actually asks for.

In [ ]:
# Cell 12 — PennyLane ±ZNE ablation (manual Richardson extrapolation: fit a degree-(n-1)
# polynomial through the (noise_scale, expectation) points and evaluate at scale=0 — for
# points at scale=0 this is exactly the fitted polynomial's constant term, so no need for
# np.polyval, just take the last coefficient)
ZNE_TEST_SUBSET = 200  # per-sample multi-scale circuit execution is expensive; subsample for this ablation
BASE_NOISE_PROB = 0.02


def richardson_extrapolate(scale_factors, values):
    coeffs = np.polyfit(scale_factors, values, deg=len(scale_factors) - 1)
    return coeffs[-1]


rng = np.random.RandomState(42)
zne_idx = rng.choice(len(test_ds), size=min(ZNE_TEST_SUBSET, len(test_ds)), replace=False)
zne_embeddings = cached["test"]["embeddings"][zne_idx]
zne_labels = cached["test"]["labels"][zne_idx].numpy()

pennylane_runs = [r for r in qhbert_results if r.get("backend") == "pennylane"]
best_pennylane = max(pennylane_runs, key=lambda r: r["f1"])
best_pl_model = trained_models[best_pennylane["run_name"]]
n_qubits_pl, n_layers_pl = best_pennylane["n_qubits"], best_pennylane["n_layers"]
trained_weights_pl = best_pl_model.quantum.qnode_weights["weights"].detach().numpy()

dev_noisy = qml.device("default.mixed", wires=n_qubits_pl)


@qml.qnode(dev_noisy)
def noisy_circuit(inputs, weights, noise_prob):
    qml.AngleEmbedding(inputs, wires=range(n_qubits_pl), rotation="Y")
    for layer in range(n_layers_pl):
        for i in range(n_qubits_pl):
            qml.CNOT(wires=[i, (i + 1) % n_qubits_pl])
            qml.DepolarizingChannel(noise_prob, wires=i)
        for qubit in range(n_qubits_pl):
            qml.RY(weights[layer, qubit, 0], wires=qubit)
            qml.RZ(weights[layer, qubit, 1], wires=qubit)
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits_pl)]


def classify_from_features(model, features_batch):
    # model.bridge/model.classifier now live on DEVICE (Cell 9's move_to_device_except_quantum) —
    # features_batch comes from the CPU-only quantum simulator, so it must move to DEVICE before
    # hitting the classifier, and the output must come back to CPU before .numpy()/further use.
    with torch.no_grad():
        features_tensor = torch.tensor(np.stack(features_batch), dtype=torch.float32).to(DEVICE)
        logits = model.classifier(features_tensor)
    return logits.argmax(dim=1).cpu().numpy()


with torch.no_grad():
    bridge_out_pl = (torch.tanh(best_pl_model.bridge(zne_embeddings.to(DEVICE))) * torch.pi).cpu().numpy()

noiseless_feats, noisy_feats, zne_feats = [], [], []
for x in bridge_out_pl:
    noiseless_feats.append(np.array(noisy_circuit(x, trained_weights_pl, 1e-9)))  # ~zero noise, same circuit path
    noisy_feats.append(np.array(noisy_circuit(x, trained_weights_pl, BASE_NOISE_PROB)))
    per_scale = [np.array(noisy_circuit(x, trained_weights_pl, BASE_NOISE_PROB * s)) for s in ZNE_NOISE_SCALE_FACTORS]
    zne_feats.append(richardson_extrapolate(ZNE_NOISE_SCALE_FACTORS, np.stack(per_scale)))

noiseless_acc = accuracy_score(zne_labels, classify_from_features(best_pl_model, noiseless_feats))
noisy_acc = accuracy_score(zne_labels, classify_from_features(best_pl_model, noisy_feats))
zne_acc = accuracy_score(zne_labels, classify_from_features(best_pl_model, zne_feats))

print(f"PennyLane ZNE ablation (best config: {best_pennylane['run_name']}, n={len(zne_idx)} test subset):")
print(f"  noiseless (grid reference, full test set) : f1={best_pennylane['f1']:.3f}")
print(f"  noiseless (this subset)                    : acc={noiseless_acc:.3f}")
print(f"  noisy, unmitigated (p={BASE_NOISE_PROB})               : acc={noisy_acc:.3f}")
print(f"  noisy + ZNE (Richardson, scales={ZNE_NOISE_SCALE_FACTORS})     : acc={zne_acc:.3f}")

zne_pennylane_results = {
    "best_config": best_pennylane["run_name"], "subset_size": len(zne_idx),
    "noiseless_acc": noiseless_acc, "noisy_acc": noisy_acc, "zne_acc": zne_acc,
    "base_noise_prob": BASE_NOISE_PROB, "scale_factors": ZNE_NOISE_SCALE_FACTORS,
}

In [ ]:
# Cell 13 — Qiskit ±ZNE ablation (same subset/scheme as Cell 12, on the best Qiskit config,
# using qiskit_aer's noisy AerEstimator — this is what closes the "qiskit_aer not installed
# yet" gap flagged in the library-check cell)
from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit_aer.primitives import EstimatorV2 as AerEstimator

qiskit_runs = [r for r in qhbert_results if r.get("backend") == "qiskit"]
best_qiskit = max(qiskit_runs, key=lambda r: r["f1"])
best_qk_model = trained_models[best_qiskit["run_name"]]
qnn_qk = best_qk_model.quantum.neural_network
circuit_qk = qnn_qk.circuit
input_params_qk = list(qnn_qk.input_params)
weight_params_qk = list(qnn_qk.weight_params)
observables_qk = qnn_qk.observables
trained_weights_qk = best_qk_model.quantum.weight.detach().numpy()


def estimate_qiskit(x, noise_prob):
    bound = circuit_qk.assign_parameters(
        dict(zip(input_params_qk, x)) | dict(zip(weight_params_qk, trained_weights_qk))
    )
    if noise_prob > 0:
        noise_model = NoiseModel()
        noise_model.add_all_qubit_quantum_error(depolarizing_error(noise_prob, 2), ["cx"])
        estimator = AerEstimator(options={"backend_options": {"noise_model": noise_model}})
    else:
        estimator = AerEstimator()
    job = estimator.run([(bound, observables_qk)])
    return np.array(job.result()[0].data.evs)


with torch.no_grad():
    bridge_out_qk = (torch.tanh(best_qk_model.bridge(zne_embeddings.to(DEVICE))) * torch.pi).cpu().numpy()

noiseless_feats_qk, noisy_feats_qk, zne_feats_qk = [], [], []
for x in bridge_out_qk:
    noiseless_feats_qk.append(estimate_qiskit(x, 0.0))
    noisy_feats_qk.append(estimate_qiskit(x, BASE_NOISE_PROB))
    per_scale = [estimate_qiskit(x, BASE_NOISE_PROB * s) for s in ZNE_NOISE_SCALE_FACTORS]
    zne_feats_qk.append(richardson_extrapolate(ZNE_NOISE_SCALE_FACTORS, np.stack(per_scale)))

noiseless_acc_qk = accuracy_score(zne_labels, classify_from_features(best_qk_model, noiseless_feats_qk))
noisy_acc_qk = accuracy_score(zne_labels, classify_from_features(best_qk_model, noisy_feats_qk))
zne_acc_qk = accuracy_score(zne_labels, classify_from_features(best_qk_model, zne_feats_qk))

print(f"Qiskit ZNE ablation (best config: {best_qiskit['run_name']}, n={len(zne_idx)} test subset):")
print(f"  noiseless (grid reference, full test set) : f1={best_qiskit['f1']:.3f}")
print(f"  noiseless (this subset)                    : acc={noiseless_acc_qk:.3f}")
print(f"  noisy, unmitigated (p={BASE_NOISE_PROB})               : acc={noisy_acc_qk:.3f}")
print(f"  noisy + ZNE (Richardson, scales={ZNE_NOISE_SCALE_FACTORS})     : acc={zne_acc_qk:.3f}")

zne_qiskit_results = {
    "best_config": best_qiskit["run_name"], "subset_size": len(zne_idx),
    "noiseless_acc": noiseless_acc_qk, "noisy_acc": noisy_acc_qk, "zne_acc": zne_acc_qk,
    "base_noise_prob": BASE_NOISE_PROB, "scale_factors": ZNE_NOISE_SCALE_FACTORS,
}

## Classical baselines

TF-IDF+SVM at full scale, plus the PyTorch `BiLSTM`/`CNN`/`Transformer`
baselines from `src/models/baselines.py` (reproduced inline here, same
reasoning as the QHBERT classes above — self-contained on Kaggle). These use
a small word-level vocab built from the training text, not DistilBERT's
tokenizer — that's the point: an independent, non-transformer-embedding
comparison point.

In [ ]:
# Cell 14 — TF-IDF + SVM baseline (full scale, full 2000-dim TF-IDF, not the 8-dim quantum bottleneck)
from sklearn.feature_extraction.text import TfidfVectorizer as _Tfidf

tfidf_full = _Tfidf(max_features=2000, stop_words="english")
X_train_tfidf_full = tfidf_full.fit_transform(X_train_text)
X_test_tfidf_full = tfidf_full.transform(X_test_text)

svc_baseline = SVC()
svc_baseline.fit(X_train_tfidf_full, y_train)
svc_preds = svc_baseline.predict(X_test_tfidf_full)

classical_results = [{
    "run_name": "tfidf_svm_full", "n_params": None, "train_time_sec": None,
    "accuracy": accuracy_score(y_test, svc_preds),
    "precision": precision_score(y_test, svc_preds),
    "recall": recall_score(y_test, svc_preds),
    "f1": f1_score(y_test, svc_preds),
    "auc_roc": None,
}]
print(f"TF-IDF+SVM: acc={classical_results[0]['accuracy']:.3f}, f1={classical_results[0]['f1']:.3f}")

In [ ]:
# Cell 15 — word-level vocab + padded token-id tensors for the BiLSTM/CNN/Transformer baselines
from collections import Counter

BASELINE_VOCAB_SIZE = 20000
BASELINE_MAX_LEN = 200

word_counts = Counter()
for text in X_train_text:
    word_counts.update(text.lower().split())

vocab = {"<pad>": 0, "<unk>": 1}
for word, _ in word_counts.most_common(BASELINE_VOCAB_SIZE - len(vocab)):
    vocab[word] = len(vocab)


def encode(text, max_len=BASELINE_MAX_LEN):
    ids = [vocab.get(w, vocab["<unk>"]) for w in text.lower().split()[:max_len]]
    ids += [vocab["<pad>"]] * (max_len - len(ids))
    return ids


def make_token_dataset(texts, labels):
    ids = torch.tensor([encode(t) for t in texts], dtype=torch.long)
    return TensorDataset(ids, torch.tensor(labels.values, dtype=torch.long))


baseline_train_ds = make_token_dataset(X_train_text, y_train)
baseline_valid_ds = make_token_dataset(X_valid_text, y_valid)
baseline_test_ds = make_token_dataset(X_test_text, y_test)
baseline_train_loader = DataLoader(baseline_train_ds, batch_size=BATCH_SIZE, shuffle=True)
baseline_valid_loader = DataLoader(baseline_valid_ds, batch_size=BATCH_SIZE)
baseline_test_loader = DataLoader(baseline_test_ds, batch_size=BATCH_SIZE)

print(f"Vocab size: {len(vocab)}, sequence length: {BASELINE_MAX_LEN}")

In [ ]:
# Cell 16 — BiLSTM/CNN/Transformer baselines (mirrors src/models/baselines.py), trained on DEVICE
# (unlike QHBERT, these have no quantum simulator to keep on CPU)


class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim=32, hidden_dim=64, num_classes=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x):
        embedded = self.embedding(x)
        _, (hidden, _) = self.lstm(embedded)
        return self.classifier(torch.cat([hidden[-2], hidden[-1]], dim=-1))


class CNNClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim=32, num_filters=128, kernel_size=5, hidden_dim=64,
                 num_classes=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.conv = nn.Conv1d(embedding_dim, num_filters, kernel_size, padding=kernel_size // 2)
        self.classifier = nn.Sequential(
            nn.ReLU(), nn.AdaptiveMaxPool1d(1), nn.Flatten(),
            nn.Linear(num_filters, hidden_dim), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.conv(self.embedding(x).transpose(1, 2)))


class TransformerEncoderClassifier(nn.Module):
    def __init__(self, vocab_size, max_len=BASELINE_MAX_LEN, embedding_dim=64, num_heads=4, ff_dim=128,
                 hidden_dim=64, num_classes=2, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.pos_embedding = nn.Embedding(max_len, embedding_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim, nhead=num_heads, dim_feedforward=ff_dim, dropout=dropout, batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=1)
        self.classifier = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x):
        positions = torch.arange(x.size(1), device=x.device).unsqueeze(0)
        encoded = self.encoder(self.embedding(x) + self.pos_embedding(positions))
        return self.classifier(encoded.mean(dim=1))


def train_baseline(model_cls, run_name, epochs=EPOCHS):
    model = model_cls(vocab_size=len(vocab)).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    best_val_f1, best_state = 0.0, None
    start = time.time()
    for epoch in range(epochs):
        model.train()
        for ids, labels in baseline_train_loader:
            ids, labels = ids.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(ids), labels)
            loss.backward()
            optimizer.step()

        model.eval()
        val_preds, val_labels = [], []
        with torch.no_grad():
            for ids, labels in baseline_valid_loader:
                val_preds.extend(model(ids.to(DEVICE)).argmax(dim=1).cpu().tolist())
                val_labels.extend(labels.tolist())
        val_f1 = f1_score(val_labels, val_preds)
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
    train_time = time.time() - start

    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    test_preds, test_probs, test_labels = [], [], []
    with torch.no_grad():
        for ids, labels in baseline_test_loader:
            logits = model(ids.to(DEVICE))
            test_preds.extend(logits.argmax(dim=1).cpu().tolist())
            test_probs.extend(torch.softmax(logits, dim=1)[:, 1].cpu().tolist())
            test_labels.extend(labels.tolist())

    print(f"[{run_name}] best val_f1={best_val_f1:.4f}, train_time={train_time:.1f}s")
    return {
        "run_name": run_name, "n_params": param_count(model), "train_time_sec": train_time,
        "accuracy": accuracy_score(test_labels, test_preds), "precision": precision_score(test_labels, test_preds),
        "recall": recall_score(test_labels, test_preds), "f1": f1_score(test_labels, test_preds),
        "auc_roc": roc_auc_score(test_labels, test_probs),
    }


for model_cls, name in [(BiLSTMClassifier, "bilstm"), (CNNClassifier, "cnn"), (TransformerEncoderClassifier, "transformer")]:
    classical_results.append(train_baseline(model_cls, name))

classical_results_df = pd.DataFrame(classical_results)
print("\nClassical baseline results:")
print(classical_results_df.to_string(index=False))

## Full BERT fine-tune + quantum layer (QHBERTFineTuneModel)

Everything above trains on **frozen** DistilBERT CLS embeddings (Cell 6, `@torch.no_grad()`) —
BERT never adapts to the fake/real-news task, only the bridge/quantum/head do. That's a valid,
cheaper design, but it isn't "BERT fine-tuned with a quantum technique" — it's a fixed feature
extractor feeding a trainable quantum head.

This section adds that: DistilBERT is **unfrozen** and trained end-to-end together with the same
`build_quantum_layer` (Cell 7) used above, so the quantum layer sits on top of representations
that can actually move during training, not a static embedding. Practical differences from the
frozen path, all driven by the fact that BERT now runs live inside the training loop every step
instead of once up front:

- No pre-cached embeddings — texts are tokenized to `input_ids`/`attention_mask` and BERT runs
  forward (and backward) on every batch.
- Smaller batch size and sequence length (`FT_BATCH_SIZE`, `FT_MAX_LENGTH` in Cell 3) to fit GPU
  memory alongside a live 66M-parameter BERT plus the quantum layer's autograd graph.
- Fewer epochs (`FT_EPOCHS=3`) — full fine-tuning converges faster and overfits an already-easy
  dataset like ISOT quickly.
- PennyLane only by default (`FT_BACKEND`) — Qiskit's SPSA gradient path is already far slower
  than PennyLane's backprop path even with cheap frozen embeddings; combined with a live BERT
  it's not practical for a full run.
- The quantum simulator is still CPU-only (see `move_to_device_except_quantum`, Cell 8) — BERT,
  the bridge, and the classifier run on GPU; the quantum layer round-trips through CPU each call.

This result is reported separately from the ablation grid above (not mixed into `qhbert_results`
or the ZNE cells, which assume frozen-embedding inputs) and merged into the final comparison
table in the last cell.

In [ ]:
# Cell FT-1 — tokenized dataset for full BERT fine-tuning. No frozen-embedding caching here:
# gradients must flow through BERT, so we cache token ids/attention masks instead (deterministic,
# cheap) and run BERT live inside the training loop.
ft_tokenizer = tokenizer  # reuse the tokenizer already loaded in Cell 6


def tokenize_split(texts):
    encoded = ft_tokenizer(list(texts), padding="max_length", truncation=True,
                            max_length=FT_MAX_LENGTH, return_tensors="pt")
    return encoded["input_ids"], encoded["attention_mask"]


ft_splits = {}
for split_name, (texts, labels) in splits_text.items():
    input_ids, attention_mask = tokenize_split(texts)
    ft_splits[split_name] = TensorDataset(input_ids, attention_mask, torch.tensor(labels.values, dtype=torch.long))

ft_train_loader = DataLoader(ft_splits["train"], batch_size=FT_BATCH_SIZE, shuffle=True)
ft_valid_loader = DataLoader(ft_splits["valid"], batch_size=FT_BATCH_SIZE)
ft_test_loader = DataLoader(ft_splits["test"], batch_size=FT_BATCH_SIZE)
print(f"Fine-tune tokenized (max_length={FT_MAX_LENGTH}): "
      f"train={len(ft_splits['train'])}, valid={len(ft_splits['valid'])}, test={len(ft_splits['test'])}")

In [ ]:
# Cell FT-2 — QHBERTFineTuneModel: same bridge/quantum/head as QHBERTModel (Cell 8), but wraps a
# trainable DistilBERT instead of consuming pre-cached frozen embeddings.
class QHBERTFineTuneModel(nn.Module):
    def __init__(self, bert_dim=768, bridge_dims=BRIDGE_DIMS, n_qubits=FT_N_QUBITS, n_layers=FT_N_LAYERS,
                 backend=FT_BACKEND, head_dim=HEAD_DIM, num_labels=2, dropout=0.3, quantum_kwargs=None):
        super().__init__()
        self.bert = DistilBertModel.from_pretrained("distilbert-base-uncased")

        bridge_layers = []
        in_dim = bert_dim
        for dim in bridge_dims:
            bridge_layers += [nn.Linear(in_dim, dim), nn.LayerNorm(dim), nn.ReLU(), nn.Dropout(dropout)]
            in_dim = dim
        bridge_layers.append(nn.Linear(in_dim, n_qubits))
        self.bridge = nn.Sequential(*bridge_layers)

        self.quantum = build_quantum_layer(backend, n_qubits, n_layers, **(quantum_kwargs or {}))

        self.classifier = nn.Sequential(
            nn.Linear(n_qubits, head_dim), nn.ReLU(), nn.Dropout(dropout), nn.Linear(head_dim, num_labels),
        )

    def forward(self, input_ids, attention_mask):
        cls_embedding = self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0, :]
        scaled = torch.tanh(self.bridge(cls_embedding)) * torch.pi
        # quantum simulator is CPU-only regardless of backend — see Cell 8's note
        features = self.quantum(scaled.cpu()).to(scaled.device)
        return self.classifier(features)


print(f"QHBERTFineTuneModel param count ({FT_BACKEND}, {FT_N_QUBITS} qubits): "
      f"{param_count(QHBERTFineTuneModel())} (vs. ~215K for the frozen-embedding QHBERTModel — "
      f"almost all of the difference is DistilBERT's ~66M trainable params)")

In [ ]:
# Cell FT-3 — train/eval loop for QHBERTFineTuneModel. Three LR groups (BERT gets a much smaller
# LR than the bridge/quantum — it's pretrained, not trained from scratch) and gradient clipping
# (full fine-tuning is more prone to loss spikes than the frozen-embedding path).
def evaluate_ft(model, loader):
    model.eval()
    all_preds, all_probs, all_labels = [], [], []
    with torch.no_grad():
        for input_ids, attention_mask, labels in loader:
            input_ids, attention_mask = input_ids.to(DEVICE), attention_mask.to(DEVICE)
            logits = model(input_ids, attention_mask)
            probs = torch.softmax(logits, dim=1)[:, 1]
            all_preds.extend(logits.argmax(dim=1).cpu().tolist())
            all_probs.extend(probs.cpu().tolist())
            all_labels.extend(labels.tolist())
    return all_preds, all_probs, all_labels


def train_and_evaluate_ft(model, run_name, epochs=FT_EPOCHS):
    move_to_device_except_quantum(model, DEVICE)
    optimizer = torch.optim.AdamW([
        {"params": model.bert.parameters(), "lr": LR_BERT},
        {"params": list(model.bridge.parameters()) + list(model.classifier.parameters()), "lr": LR_CLASSICAL},
        {"params": model.quantum.parameters(), "lr": LR_QUANTUM},
    ])
    criterion = nn.CrossEntropyLoss()

    best_val_f1, best_state = 0.0, None
    start = time.time()
    for epoch in range(epochs):
        model.train()
        total_loss, n_seen = 0.0, 0
        for input_ids, attention_mask, labels in ft_train_loader:
            input_ids, attention_mask, labels = input_ids.to(DEVICE), attention_mask.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(input_ids, attention_mask), labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item() * input_ids.size(0)
            n_seen += input_ids.size(0)

        val_preds, _, val_labels = evaluate_ft(model, ft_valid_loader)
        val_f1 = f1_score(val_labels, val_preds)
        print(f"[{run_name}] epoch {epoch + 1}/{epochs} — loss {total_loss / n_seen:.4f}, val_f1 {val_f1:.4f}")
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
    train_time = time.time() - start

    if best_state is not None:
        model.load_state_dict(best_state)
    test_preds, test_probs, test_labels = evaluate_ft(model, ft_test_loader)
    return {
        "run_name": run_name,
        "n_params": param_count(model),
        "train_time_sec": train_time,
        "accuracy": accuracy_score(test_labels, test_preds),
        "precision": precision_score(test_labels, test_preds),
        "recall": recall_score(test_labels, test_preds),
        "f1": f1_score(test_labels, test_preds),
        "auc_roc": roc_auc_score(test_labels, test_probs),
        "backend": FT_BACKEND,
        "n_qubits": FT_N_QUBITS,
        "n_layers": FT_N_LAYERS,
        "train_rows": len(ft_splits["train"]),
        "bert_frozen": False,
    }

In [ ]:
# Cell FT-4 — time 1 epoch before committing (same reasoning as Cell 10: full BERT fine-tuning is
# much heavier than the frozen-embedding path, confirm the real cost before running FT_EPOCHS).
_ft_sanity_model = QHBERTFineTuneModel()
_t0 = time.time()
_ = train_and_evaluate_ft(_ft_sanity_model, run_name="sanity_check_finetune", epochs=1)
_ft_epoch_time = time.time() - _t0
print(f"-> {_ft_epoch_time:.1f}s/epoch, projects to ~{_ft_epoch_time * FT_EPOCHS / 60:.1f} min for "
      f"{FT_EPOCHS} epochs. If too slow for your remaining session time, shrink FT_EPOCHS, "
      f"FT_MAX_LENGTH, or FT_BATCH_SIZE in Cell 3 before running the cell below.")
del _ft_sanity_model
torch.cuda.empty_cache() if DEVICE.type == "cuda" else None

In [ ]:
# Cell FT-5 — the actual full-fine-tune run
qhbert_finetune_model = QHBERTFineTuneModel()
qhbert_finetune_result = train_and_evaluate_ft(
    qhbert_finetune_model, run_name=f"qhbert_finetuned_bert_{FT_BACKEND}_{FT_N_QUBITS}q_{FT_N_LAYERS}L"
)
print("\nFull fine-tune result:")
print(qhbert_finetune_result)

In [ ]:
# Cell 17 — final combined table (QHBERT ablation grid + full fine-tune + classical baselines +
# Tier 1/2 recap) + save
combined_df = pd.concat(
    [qhbert_results_df, pd.DataFrame([qhbert_finetune_result]), classical_results_df], ignore_index=True
)
print("=== Combined comparison (all methods, ISOT, this notebook's split) ===")
print(combined_df[["run_name", "n_params", "accuracy", "precision", "recall", "f1", "auc_roc", "train_time_sec"]]
      .to_string(index=False))

if tier12_results is not None:
    print("\n=== Tier 1/2 recap (qiskit-practice.ipynb, TF-IDF+SVD features, different feature pipeline) ===")
    print(f"  VQC (8-dim SVD, n={tier12_results['vqc']['train_size']} subsample): "
          f"test_acc={tier12_results['vqc']['test_acc']:.3f}")
    print(f"  SVC (same 8-dim features)                     : "
          f"test_acc={tier12_results['svc_reduced']['test_acc']:.3f}")

final_results = {
    "dataset": "isot",
    "label_convention": "0=fake, 1=real",
    "qhbert_and_classical": combined_df.to_dict(orient="records"),
    "zne_pennylane": zne_pennylane_results,
    "zne_qiskit": zne_qiskit_results,
    "tier1_tier2_qiskit_practice_recap": tier12_results,
}
with open("qhbert_full_comparison_results.json", "w") as f:
    json.dump(final_results, f, indent=2, default=str)
print("\nSaved qhbert_full_comparison_results.json")